# Multi-domain Stage 3 — unchanged continuation, step 100 to 130

Loads the newest verified full-state checkpoint in [100, 130], keeps the corrected fan data and all optimization settings unchanged, and stops after the step-130 gate.


In [1]:
%pip install -q transformers==5.13.1 peft==0.19.1 bitsandbytes==0.50.0 accelerate


In [2]:
import gc, hashlib, json, logging, math, os, random, re
from collections import Counter
from pathlib import Path
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import numpy as np
import torch
from torch.utils.data import Dataset
from google.colab import drive
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainerCallback, TrainingArguments

drive.mount('/content/drive',force_remount=False)
if not torch.cuda.is_available(): raise RuntimeError('A Colab GPU is required.')
SEED=20260812; MODEL_NAME='Qwen/Qwen2.5-3B-Instruct'
RUN_DIR=Path('/content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1')
TRAINER_DIR=RUN_DIR/'trainer-output'; EVENT_LOG=RUN_DIR/'first_gate_report.json'
STEP100_REPORT=RUN_DIR/'step100_fan_fix_report.json'
STEP130_REPORT=RUN_DIR/'step130_report.json'
EVAL_PROGRESS=RUN_DIR/'step130_eval_progress.json'
MAX_STEPS=150; SOURCE_STEP=100; TARGET_STEP=130; MAX_SEQ_LENGTH=1024; MAX_NEW_TOKENS=300
RUN_DIR.mkdir(parents=True,exist_ok=True)

def atomic_json(path,payload):
    temp=path.with_suffix(path.suffix+'.tmp'); temp.write_text(json.dumps(payload,indent=2,sort_keys=True)); temp.replace(path)
def valid_checkpoint(path):
    return (path.is_dir() and all((path/x).is_file() and (path/x).stat().st_size>0
                                  for x in ('trainer_state.json','optimizer.pt','scheduler.pt'))
            and any(p.name.startswith('adapter_model') and p.stat().st_size>0 for p in path.iterdir()))
candidates=[]
if TRAINER_DIR.is_dir():
    for path in TRAINER_DIR.glob('checkpoint-*'):
        try: step=int(path.name.rsplit('-',1)[1])
        except ValueError: continue
        if valid_checkpoint(path): candidates.append((step,path))
RESUME_STEP,RESUME_CHECKPOINT=max(candidates,default=(0,None),key=lambda x:x[0])
if not STEP100_REPORT.is_file() or not json.loads(STEP100_REPORT.read_text()).get('step100_complete'):
    raise RuntimeError(f'Missing completed step-100 report: {STEP100_REPORT}')
if STEP130_REPORT.is_file() and json.loads(STEP130_REPORT.read_text()).get('step130_complete'):
    completed=json.loads(STEP130_REPORT.read_text())
    raise RuntimeError(f'Step 130 is already complete at {completed.get("checkpoint")}. Inspect the report; do not retrain.')
if not SOURCE_STEP<=RESUME_STEP<=TARGET_STEP:
    raise RuntimeError(f'Expected a valid checkpoint from step 100 through 130, found: {RESUME_CHECKPOINT}')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print({'gpu':torch.cuda.get_device_name(0),'resume_checkpoint':str(RESUME_CHECKPOINT),
       'loaded_state':['adapter','optimizer','scheduler','trainer','RNG'],
       'authorized_stop':TARGET_STEP,'scheduler_horizon':MAX_STEPS})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'gpu': 'NVIDIA L4', 'resume_checkpoint': '/content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-100', 'loaded_state': ['adapter', 'optimizer', 'scheduler', 'trainer', 'RNG'], 'authorized_stop': 130, 'scheduler_horizon': 150}


In [3]:
"""Deterministic multi-domain binary-state seeding corpus and verifier."""

from collections import Counter
from dataclasses import asdict, dataclass
import itertools
import random
import re
from typing import Sequence


DEFAULT_SEED = 20260812
OPERATIONS = ("same", "different")
DOMAIN_SPECS = {
    "fan": {
        "states": ("Running", "Stopped"),
        "subject": "A greenhouse ventilation fan",
        "same": "same as previously (the fan mode does NOT change)",
        "different": "different from previously (the fan changes to its other mode)",
        "same_reason": "The fan mode stays the same",
        "different_reason": "The fan changes mode",
    },
    "valve": {
        "states": ("Open", "Closed"),
        "subject": "An irrigation water valve",
        "same": "same as previously (the valve position does NOT change)",
        "different": "different from previously (the valve moves to its other position)",
        "same_reason": "The valve position stays the same",
        "different_reason": "The valve changes position",
    },
    "lamp": {
        "states": ("Lit", "Dark"),
        "subject": "A laboratory signal lamp",
        "same": "same as previously (the lamp condition does NOT change)",
        "different": "different from previously (the lamp changes to its other condition)",
        "same_reason": "The lamp condition stays the same",
        "different_reason": "The lamp changes condition",
    },
}
TOKEN_RE = re.compile(r"^[A-Z][A-Za-z]{2,9}$")


@dataclass(frozen=True)
class MultiDomainExample:
    example_id: str
    split: str
    domain: str
    initial_state: str
    operations: tuple[str, ...]
    expected_states: tuple[str, ...]
    token_for_first_state: str
    token_for_second_state: str
    prompt: str
    demonstration: str
    final_answer: str

    def mapping(self) -> dict[str, str]:
        states = DOMAIN_SPECS[self.domain]["states"]
        return {states[0]: self.token_for_first_state, states[1]: self.token_for_second_state}

    def to_dict(self) -> dict[str, object]:
        row = asdict(self)
        row["operations"] = list(self.operations)
        row["expected_states"] = list(self.expected_states)
        return row


def simulate(domain: str, initial_state: str, operations: Sequence[str]) -> list[str]:
    states = DOMAIN_SPECS[domain]["states"]
    state = initial_state
    output = []
    for operation in operations:
        if operation == "different":
            state = states[1] if state == states[0] else states[0]
        elif operation != "same":
            raise ValueError(operation)
        output.append(state)
    return output


def _signatures(domain: str, rng: random.Random, per_cell: int):
    states = DOMAIN_SPECS[domain]["states"]
    cells = {(initial, final): [] for initial in states for final in states}
    for length in range(3, 9):
        for initial in states:
            for operations in itertools.product(OPERATIONS, repeat=length):
                final = simulate(domain, initial, operations)[-1]
                cells[(initial, final)].append((initial, operations))
    selected = []
    for key in sorted(cells):
        rng.shuffle(cells[key]); selected.extend(cells[key][:per_cell])
    rng.shuffle(selected)
    return selected


def _train_eval_signatures(domain: str, rng: random.Random, train_per_cell: int, eval_per_cell: int):
    states = DOMAIN_SPECS[domain]["states"]
    cells = {(initial, final): [] for initial in states for final in states}
    for length in range(3, 9):
        for initial in states:
            for operations in itertools.product(OPERATIONS, repeat=length):
                final = simulate(domain, initial, operations)[-1]
                cells[(initial, final)].append((initial, operations))
    training, evaluation = [], []
    for key in sorted(cells):
        rng.shuffle(cells[key])
        training.extend(cells[key][:train_per_cell])
        evaluation.extend(cells[key][train_per_cell:train_per_cell + eval_per_cell])
    rng.shuffle(training); rng.shuffle(evaluation)
    return training, evaluation


def _nonce_stream(rng: random.Random):
    consonants, vowels = "bcdfghjklmnprstvwxyz", "aeiou"
    banned = ("run", "stop", "open", "close", "lit", "dark", "fan", "valve", "lamp",
              "lock", "head", "tail", "coin")
    seen = set()
    while True:
        token = "".join(rng.choice(consonants) + rng.choice(vowels)
                        for _ in range(rng.choice((2, 3, 4)))).capitalize()
        if token in seen or any(word in token.casefold() for word in banned): continue
        seen.add(token); yield token


def _render(domain, initial, operations, expected, mapping, corrected_fan_wording=False):
    spec = DOMAIN_SPECS[domain]; states = spec["states"]
    different_instruction = spec["different"]
    different_reason = spec["different_reason"]
    if domain == "fan" and corrected_fan_wording:
        different_instruction = (
            "different from previously (the fan flips to the opposite state: "
            "Running becomes Stopped, and Stopped becomes Running)"
        )
        different_reason = "The fan flips to its opposite state"
    prompt_lines = [
        f"{spec['subject']} starts {initial}.",
        "Track its physical state through every instruction.",
        f"Represent {states[0]} using the code {mapping[states[0]]} and {states[1]} using the code {mapping[states[1]]}.",
        f"Therefore, the initial code is {mapping[initial]}.",
        *[f"{i}. {spec['same'] if op == 'same' else different_instruction}"
          for i, op in enumerate(operations, 1)],
        "Apply the declared mapping in every numbered step.",
        "Write each line as: Step i: <brief reasoning>. State: <declared code word>",
        "After all steps, write exactly: Final coded state: <code>. <code> represents <physical state>.",
        "End with the physical state, not its code, inside <answer>...</answer>.",
    ]
    demo_lines = [
        f"Step {i}: {spec['same_reason'] if op == 'same' else different_reason}. State: {mapping[state]}"
        for i, (op, state) in enumerate(zip(operations, expected), 1)
    ]
    final = expected[-1]; token = mapping[final]
    demo_lines.extend([f"Final coded state: {token}. {token} represents {final}.",
                       f"<answer>{final}</answer>"])
    return "\n".join(prompt_lines), "\n".join(demo_lines)


def generate_multidomain_dataset(seed: int = DEFAULT_SEED, corrected_fan_wording: bool = False):
    rng = random.Random(seed); nonces = _nonce_stream(rng)
    training, evaluations = [], {domain: [] for domain in DOMAIN_SPECS}
    for domain in ("fan", "valve"):
        train_signatures, eval_signatures = _train_eval_signatures(domain, rng, 100, 25)
        for split, signatures in (("train", train_signatures), ("eval", eval_signatures)):
            target = training if split == "train" else evaluations[domain]
            for index, (initial, operations) in enumerate(signatures):
                states = DOMAIN_SPECS[domain]["states"]
                first, second = next(nonces), next(nonces)
                if index % 2: first, second = second, first
                mapping = {states[0]: first, states[1]: second}
                expected = tuple(simulate(domain, initial, operations))
                prompt, demo = _render(
                    domain, initial, operations, expected, mapping, corrected_fan_wording
                )
                target.append(MultiDomainExample(
                    f"{domain}-{split}-{index:04d}", split, domain, initial, operations,
                    expected, first, second, prompt, demo, expected[-1]))
    domain = "lamp"
    for index, (initial, operations) in enumerate(_signatures(domain, rng, 25)):
        states = DOMAIN_SPECS[domain]["states"]; first, second = next(nonces), next(nonces)
        if index % 2: first, second = second, first
        mapping = {states[0]: first, states[1]: second}
        expected = tuple(simulate(domain, initial, operations))
        prompt, demo = _render(domain, initial, operations, expected, mapping, False)
        evaluations[domain].append(MultiDomainExample(
            f"lamp-eval-{index:04d}", "eval", domain, initial, operations,
            expected, first, second, prompt, demo, expected[-1]))
    # Exact alternation ensures domain interleaving before Trainer shuffling.
    fan = [row for row in training if row.domain == "fan"]
    valve = [row for row in training if row.domain == "valve"]
    interleaved = [row for pair in zip(fan, valve) for row in pair]
    return interleaved, evaluations


def verify_example(row: MultiDomainExample):
    errors = []; spec = DOMAIN_SPECS[row.domain]; states = spec["states"]; mapping = row.mapping()
    expected = tuple(simulate(row.domain, row.initial_state, row.operations))
    if expected != row.expected_states: errors.append("wrong_expected_states")
    declaration = f"Represent {states[0]} using the code {mapping[states[0]]} and {states[1]} using the code {mapping[states[1]]}."
    if row.prompt.count(declaration) != 1: errors.append("mapping_declaration")
    if f"Therefore, the initial code is {mapping[row.initial_state]}." not in row.prompt:
        errors.append("initial_anchor")
    lines = row.demonstration.splitlines(); trace = lines[:len(row.operations)]
    if len(trace) != len(row.operations): errors.append("trace_count")
    for i, (line, state) in enumerate(zip(trace, expected), 1):
        match = re.fullmatch(rf"Step {i}: .*\. State: ([A-Z][A-Za-z]{{2,9}})", line)
        if not match or match.group(1) != mapping[state]: errors.append(f"trace_{i}")
    final = expected[-1]; token = mapping[final]
    if lines[-2:] != [f"Final coded state: {token}. {token} represents {final}.",
                      f"<answer>{final}</answer>"]:
        errors.append("decode_back_or_answer")
    return not errors, errors


def audit_dataset(training, evaluations):
    all_rows = list(training) + [row for rows in evaluations.values() for row in rows]
    failures = []
    for row in all_rows:
        valid, errors = verify_example(row)
        if not valid: failures.append({"example_id": row.example_id, "errors": errors})
    report = {
        "train_count": len(training),
        "train_domains": dict(Counter(row.domain for row in training)),
        "evaluation_domains": {domain: len(rows) for domain, rows in evaluations.items()},
        "decode_back_present_count": sum("Final coded state:" in row.demonstration for row in training),
        "decode_back_training_coverage": sum("Final coded state:" in row.demonstration for row in training)/len(training),
        "semantic_pass_rate": 100*(len(all_rows)-len(failures))/len(all_rows),
        "failures": failures,
    }
    report["accepted"] = (report["train_count"] == 800
                          and report["train_domains"] == {"fan": 400, "valve": 400}
                          and report["evaluation_domains"] == {"fan": 100, "valve": 100, "lamp": 100}
                          and report["decode_back_training_coverage"] == 1.0
                          and not failures)
    return report


In [4]:
training_examples,evaluation_sets=generate_multidomain_dataset(SEED,corrected_fan_wording=True)
dataset_audit=audit_dataset(training_examples,evaluation_sets)
assert dataset_audit['accepted'] and dataset_audit['semantic_pass_rate']==100.0
assert dataset_audit['decode_back_training_coverage']==1.0
assert all(training_examples[i].domain!=training_examples[i+1].domain
           for i in range(len(training_examples)-1))
print('MULTI-DOMAIN DATA ACCEPTANCE GATE:',json.dumps(dataset_audit,indent=2,sort_keys=True))
print('CONFIRMED: every one of 800 training targets has a full decode-back line.')

old_training,old_evaluations=generate_multidomain_dataset(SEED,corrected_fan_wording=False)
assert len(old_training)==len(training_examples)==800
localized_changes=0
for old,new in zip(old_training,training_examples):
    assert (old.example_id,old.domain,old.initial_state,old.operations,old.expected_states,
            old.token_for_first_state,old.token_for_second_state)==(
            new.example_id,new.domain,new.initial_state,new.operations,new.expected_states,
            new.token_for_first_state,new.token_for_second_state)
    if old.domain=='valve': assert old==new
    elif old.prompt!=new.prompt: localized_changes+=1
assert old_evaluations['valve']==evaluation_sets['valve']
assert old_evaluations['lamp']==evaluation_sets['lamp']
assert localized_changes>0
print({'fan_only_wording_changes':localized_changes,'valve_records_unchanged':True,
       'lamp_eval_records_unchanged':True,'signatures_tokens_order_unchanged':True})


MULTI-DOMAIN DATA ACCEPTANCE GATE: {
  "accepted": true,
  "decode_back_present_count": 800,
  "decode_back_training_coverage": 1.0,
  "evaluation_domains": {
    "fan": 100,
    "lamp": 100,
    "valve": 100
  },
  "failures": [],
  "semantic_pass_rate": 100.0,
  "train_count": 800,
  "train_domains": {
    "fan": 400,
    "valve": 400
  }
}
CONFIRMED: every one of 800 training targets has a full decode-back line.
{'fan_only_wording_changes': 395, 'valve_records_unchanged': True, 'lamp_eval_records_unchanged': True, 'signatures_tokens_order_unchanged': True}


In [5]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=False)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
SYSTEM='You solve binary-state tracking tasks accurately and follow the requested output format.'
def messages(row): return [{'role':'system','content':SYSTEM},{'role':'user','content':row.prompt}]
def encode(row):
    full=tokenizer.apply_chat_template(messages(row)+[{'role':'assistant','content':row.demonstration}],tokenize=False,add_generation_prompt=False)
    start=full.rfind(row.demonstration)
    if start<0: raise RuntimeError('Assistant target missing from rendered chat.')
    encoded=tokenizer(full,add_special_tokens=False,return_offsets_mapping=True)
    labels=[token if end>start and end>begin else -100 for token,(begin,end) in zip(encoded['input_ids'],encoded['offset_mapping'])]
    if len(labels)>MAX_SEQ_LENGTH: raise RuntimeError(f'Example exceeds {MAX_SEQ_LENGTH} tokens.')
    return {'input_ids':encoded['input_ids'],'attention_mask':[1]*len(labels),'labels':labels}
class Encoded(Dataset):
    def __init__(self,rows): self.rows=[encode(row) for row in rows]
    def __len__(self): return len(self.rows)
    def __getitem__(self,index): return self.rows[index]
class Collator:
    def __call__(self,features):
        width=max(len(row['input_ids']) for row in features); ids=[]; masks=[]; labels=[]
        for row in features:
            pad=width-len(row['input_ids']); ids.append([tokenizer.pad_token_id]*pad+row['input_ids'])
            masks.append([0]*pad+row['attention_mask']); labels.append([-100]*pad+row['labels'])
        return {'input_ids':torch.tensor(ids),'attention_mask':torch.tensor(masks),'labels':torch.tensor(labels)}
train_dataset=Encoded(training_examples); collator=Collator()
target_lengths=[sum(label!=-100 for label in row['labels']) for row in train_dataset.rows]
full_lengths=[len(row['input_ids']) for row in train_dataset.rows]
assert max(target_lengths)<=math.floor(MAX_NEW_TOKENS*.8), (max(target_lengths),MAX_NEW_TOKENS)
assert max(full_lengths)<=MAX_SEQ_LENGTH
print({'train_examples':len(train_dataset),'max_target_tokens':max(target_lengths),
       'max_full_tokens':max(full_lengths),'decode_back_coverage':dataset_audit['decode_back_training_coverage']})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{'train_examples': 800, 'max_target_tokens': 180, 'max_full_tokens': 506, 'decode_back_coverage': 1.0}


In [6]:
gc.collect(); torch.cuda.empty_cache()
free,total=torch.cuda.mem_get_info()
if free/1024**3<12: raise RuntimeError(f'Only {free/1024**3:.2f} GiB GPU memory free; restart runtime.')
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.bfloat16)
base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,dtype=torch.bfloat16,quantization_config=quant,device_map={'':0},low_cpu_mem_usage=True,use_safetensors=True,trust_remote_code=False)
base.config.use_cache=False; base=prepare_model_for_kbit_training(base,use_gradient_checkpointing=True)
model=get_peft_model(base,LoraConfig(r=8,lora_alpha=16,target_modules=['q_proj','k_proj','v_proj','o_proj'],lora_dropout=.05,bias='none',task_type='CAUSAL_LM'))
assert all('lora_' in name for name,p in model.named_parameters() if p.requires_grad)
model.print_trainable_parameters(); print('FRESH MULTI-DOMAIN ADAPTER VERIFIED.')


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
FRESH MULTI-DOMAIN ADAPTER VERIFIED.


In [7]:
STEP_RE=re.compile(r'^Step\s+(\d+):.*?State:\s*([A-Z][A-Za-z]{2,9})[.,]?\s*$',re.MULTILINE)
RAW_ANSWER_RE=re.compile(r'<answer>\s*([^<\n]+?)\s*</answer>\s*$',re.IGNORECASE)
DECODE_RE=re.compile(r'^Final coded state:\s*([A-Z][A-Za-z]{2,9})\.\s*([A-Z][A-Za-z]{2,9}) represents ([A-Za-z]+)\.\s*$',re.MULTILINE)
ALL_PHYSICAL={state.casefold() for spec in DOMAIN_SPECS.values() for state in spec['states']}
def score(row,text):
    matches=[(int(i),token.casefold()) for i,token in STEP_RE.findall(text.split('<answer>',1)[0])]
    tokens=[token for _,token in matches]; declared={state:token.casefold() for state,token in row.mapping().items()}
    structural=(len(matches)==len(row.operations) and [i for i,_ in matches]==list(range(1,len(row.operations)+1)))
    internal=False; global_consistent=False; mapping_adherence=False
    if structural:
        internal=all((tokens[i]==tokens[i-1]) if row.operations[i]=='same' else (tokens[i]!=tokens[i-1]) for i in range(1,len(tokens)))
        by_state={state:set() for state in DOMAIN_SPECS[row.domain]['states']}
        for state,token in zip(row.expected_states,tokens): by_state[state].add(token)
        global_consistent=(all(len(by_state[state])==1 for state in by_state)
                           and len({next(iter(values)) for values in by_state.values()})==2)
        mapping_adherence=all(token==declared[state] for state,token in zip(row.expected_states,tokens))
    raw_match=RAW_ANSWER_RE.search(text); raw=raw_match.group(1).strip().casefold() if raw_match else None
    answer_correct=raw==row.final_answer.casefold()
    declared_codes=set(declared.values())
    answer_is_code=bool(not answer_correct and raw and raw not in ALL_PHYSICAL and (raw in set(tokens) or raw in declared_codes))
    decode_matches=DECODE_RE.findall(text)
    final_token=declared[row.final_answer]
    decode_back_correct=(len(decode_matches)==1 and decode_matches[0][0].casefold()==final_token
                         and decode_matches[0][1].casefold()==final_token
                         and decode_matches[0][2].casefold()==row.final_answer.casefold())
    if answer_correct: failure=None
    elif answer_is_code: failure='code_word_instead_of_physical_state'
    elif not structural or not internal: failure='wrong_tracking'
    elif not mapping_adherence: failure='wrong_mapping'
    elif raw is None: failure='malformed_or_missing_answer'
    else: failure='other_wrong_answer'
    return {'structural':structural,'transition_tracking':internal,'global_consistent':global_consistent,
            'mapping_adherence':mapping_adherence,'nonliteral':structural and all(t not in ALL_PHYSICAL for t in tokens),
            'decode_back_correct':decode_back_correct,'answer_correct':answer_correct,
            'answer_is_code_word':answer_is_code,'answer_failure_type':failure,
            'declared_pair':sorted(declared_codes),'text':text}

def adapter_fingerprint(model):
    digest=hashlib.sha256()
    for name,tensor in model.state_dict().items():
        if 'lora_' in name: digest.update(name.encode()); digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def aggregate(rows):
    n=len(rows); failures=[row for row in rows if not row['answer_correct']]
    pairs=Counter(tuple(row['declared_pair']) for row in rows if row['mapping_adherence'])
    return {'count':n,'physical_final_answer_accuracy':sum(r['answer_correct'] for r in rows)/n,
            'structural_format_rate':sum(r['structural'] for r in rows)/n,
            'transition_tracking_rate':sum(r['transition_tracking'] for r in rows)/n,
            'global_mapping_consistency_rate':sum(r['global_consistent'] for r in rows)/n,
            'declared_mapping_adherence_rate':sum(r['mapping_adherence'] for r in rows)/n,
            'nonliteral_encoding_rate':sum(r['nonliteral'] for r in rows)/n,
            'decode_back_specific_accuracy':sum(r['decode_back_correct'] for r in rows)/n,
            'answer_is_code_word_rate':sum(r['answer_is_code_word'] for r in rows)/n,
            'answer_is_code_word_fraction_of_failures':(sum(r['answer_is_code_word'] for r in failures)/len(failures) if failures else 0),
            'answer_failure_category_counts':dict(Counter(r['answer_failure_type'] for r in failures)),
            'distinct_correctly_applied_token_pairs':len(pairs)}

@torch.inference_mode()
def evaluate_first_gate(model):
    fingerprint=adapter_fingerprint(model)
    progress=json.loads(EVAL_PROGRESS.read_text()) if EVAL_PROGRESS.is_file() else {'fingerprint':fingerprint,'rows':[]}
    if progress['fingerprint']!=fingerprint: raise RuntimeError('Saved evaluation belongs to different adapter weights.')
    completed={(r['domain'],r['example_id']) for r in progress['rows']}
    prior_cache=model.config.use_cache; model.config.use_cache=True; model.eval()
    try:
        for domain,examples in evaluation_sets.items():
            for start in range(0,len(examples),2):
                chunk=[row for row in examples[start:start+2] if (domain,row.example_id) not in completed]
                if not chunk: continue
                prompts=[tokenizer.apply_chat_template(messages(row),tokenize=False,add_generation_prompt=True) for row in chunk]
                batch=tokenizer(prompts,return_tensors='pt',padding=True).to(model.device)
                output=model.generate(**batch,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
                texts=tokenizer.batch_decode(output[:,batch['input_ids'].shape[1]:],skip_special_tokens=True)
                for row,text in zip(chunk,texts):
                    progress['rows'].append({'domain':domain,'example_id':row.example_id,**score(row,text)})
                    completed.add((domain,row.example_id))
                if len(completed)%10==0: atomic_json(EVAL_PROGRESS,progress); print(f'SAVED STEP-20 EVAL: {len(completed)}/300')
        atomic_json(EVAL_PROGRESS,progress)
    finally: model.config.use_cache=prior_cache; model.train(); gc.collect(); torch.cuda.empty_cache()
    assert len(progress['rows'])==300
    per_domain={domain:aggregate([r for r in progress['rows'] if r['domain']==domain]) for domain in evaluation_sets}
    pooled=aggregate(progress['rows'])
    return {'per_domain':per_domain,'pooled':pooled,'samples':progress['rows'][:12]}


In [8]:
step100_report=json.loads(STEP100_REPORT.read_text())
step100_metrics=step100_report['metrics']
report={'stage':'multi_domain_stage3_step130_unchanged','source_checkpoint':str(RESUME_CHECKPOINT),
        'step130_complete':False,'target_step':130,'full_scheduler_horizon':150,
        'change':'none; corrected fan wording retained','step100_reference':step100_metrics,
        'stage4_authorized':False}

def finish_step130(model,checkpoint):
    metrics=evaluate_first_gate(model)
    tracked=('physical_final_answer_accuracy','structural_format_rate','transition_tracking_rate',
             'global_mapping_consistency_rate','declared_mapping_adherence_rate',
             'decode_back_specific_accuracy','nonliteral_encoding_rate')
    trajectory={}
    for domain in ('fan','valve','lamp'):
        before=step100_metrics['per_domain'][domain]; after=metrics['per_domain'][domain]
        trajectory[domain]={'changes_points':{key:100*(after[key]-before[key]) for key in tracked},
                            'step100_tracking':before['transition_tracking_rate'],
                            'step130_tracking':after['transition_tracking_rate']}
    fan=metrics['per_domain']['fan']; valve=metrics['per_domain']['valve']; lamp=metrics['per_domain']['lamp']
    gaps={'fan_valve_tracking_gap_points':100*abs(fan['transition_tracking_rate']-valve['transition_tracking_rate']),
          'fan_lamp_tracking_gap_points':100*abs(fan['transition_tracking_rate']-lamp['transition_tracking_rate'])}
    gaps['both_within_15_points']=all(value<=15 for key,value in gaps.items() if key.endswith('_points'))
    pooled=metrics['pooled']
    acceptance={'threshold':.95,
                'per_metric':{key:{'value':pooled[key],'passed':pooled[key]>=.95} for key in tracked},
                'zero_code_copy':pooled['answer_is_code_word_rate']==0}
    acceptance['full_pooled_gate_passed']=(all(v['passed'] for v in acceptance['per_metric'].values())
                                            and acceptance['zero_code_copy'])
    report.update({'step130_complete':True,'checkpoint':str(checkpoint),'metrics':metrics,
                   'step100_to130_trajectory':trajectory,'tracking_gaps':gaps,
                   'pooled_acceptance_gate':acceptance,'stop_reason':'step130_complete_needs_review'})
    atomic_json(STEP130_REPORT,report)
    print('===== STEP 130 STANDARD METRICS ====='); print(json.dumps(metrics,indent=2,sort_keys=True))
    print('===== STEP 100 TO 130 TRAJECTORY ====='); print(json.dumps(trajectory,indent=2,sort_keys=True))
    print('===== TRACKING GAPS ====='); print(json.dumps(gaps,indent=2,sort_keys=True))
    print('===== POOLED 95% ACCEPTANCE GATE ====='); print(json.dumps(acceptance,indent=2,sort_keys=True))

class Step130Gate(TrainerCallback):
    def on_log(self,args,state,control,logs=None,**kwargs):
        for key in ('loss','grad_norm'):
            if key in (logs or {}) and not math.isfinite(float(logs[key])):
                report['stop_reason']=f'nonfinite_{key}'
                atomic_json(STEP130_REPORT,report)
                control.should_training_stop=True
        return control
    def on_step_end(self,args,state,control,**kwargs):
        if int(state.global_step)>=TARGET_STEP:
            control.should_save=True; control.should_training_stop=True
        return control
    def on_save(self,args,state,control,model=None,**kwargs):
        step=int(state.global_step); checkpoint=Path(args.output_dir)/f'checkpoint-{step}'
        if not valid_checkpoint(checkpoint): raise RuntimeError(f'Incomplete checkpoint: {checkpoint}')
        print('VERIFIED RESUMABLE CHECKPOINT:',checkpoint)
        if step==TARGET_STEP and not report.get('step130_complete'):
            finish_step130(model,checkpoint); control.should_training_stop=True
        return control

# Preserve the original 150-step scheduler and restore its optimizer/scheduler state.
args=TrainingArguments(output_dir=str(TRAINER_DIR),per_device_train_batch_size=1,gradient_accumulation_steps=16,
    learning_rate=2e-5,lr_scheduler_type='cosine',warmup_steps=10,max_steps=MAX_STEPS,
    optim='paged_adamw_8bit',weight_decay=0.01,max_grad_norm=1.0,gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False},bf16=True,logging_steps=1,logging_first_step=True,
    logging_nan_inf_filter=False,save_strategy='steps',save_steps=5,save_total_limit=6,save_only_model=False,
    report_to='none',disable_tqdm=False,seed=SEED,data_seed=SEED,remove_unused_columns=False)
trainer=Trainer(model=model,args=args,train_dataset=train_dataset,data_collator=collator,callbacks=[Step130Gate()])
print({'resume_from':str(RESUME_CHECKPOINT),'resume_step':RESUME_STEP,'target_step':TARGET_STEP,
       'state_restored':['adapter','optimizer','scheduler','trainer','RNG'],'settings_changed':False})
result=trainer.train(resume_from_checkpoint=str(RESUME_CHECKPOINT))
if int(trainer.state.global_step)==TARGET_STEP and not report.get('step130_complete'):
    finish_step130(model,TRAINER_DIR/'checkpoint-130')
print('FINAL STATUS:',{'step':int(trainer.state.global_step),'step130_complete':report.get('step130_complete'),
                       'checkpoint':report.get('checkpoint'),
                       'full_gate':report.get('pooled_acceptance_gate',{}).get('full_pooled_gate_passed'),
                       'stage4_authorized':False})
print('STOP HERE. No Stage 3.5 or Stage 4 was executed.')


{'resume_from': '/content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-100', 'resume_step': 100, 'target_step': 130, 'state_restored': ['adapter', 'optimizer', 'scheduler', 'trainer', 'RNG'], 'settings_changed': False}


Step,Training Loss
101,0.050324
102,0.050081
103,0.045810
104,0.044682
105,0.041265
106,0.052804
107,0.049297
108,0.043857
109,0.043940
110,0.044168


VERIFIED RESUMABLE CHECKPOINT: /content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-105
VERIFIED RESUMABLE CHECKPOINT: /content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-110
VERIFIED RESUMABLE CHECKPOINT: /content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-115
VERIFIED RESUMABLE CHECKPOINT: /content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-120
VERIFIED RESUMABLE CHECKPOINT: /content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-125
VERIFIED RESUMABLE CHECKPOINT: /content/drive/MyDrive/AISI/checkpoints/multidomain-stage3-v1/trainer-output/checkpoint-130
SAVED STEP-20 EVAL: 10/300
SAVED STEP-20 EVAL: 20/300
SAVED STEP-20 EVAL: 30/300
SAVED STEP-20 EVAL: 40/300
SAVED STEP-20 EVAL: 50/300
SAVED STEP-20 EVAL: 60/300
SAVED STEP-20 EVAL: 70/300
SAVED STEP-20 EVAL: 80/300
SAVED STEP-20 EVAL: 90/300
SAVED STEP-20 EVAL: